# cFos Stressor Notebook
Adapted from the original drug analysis pipeline.

**Stressors:** Ctrl (Control), FS (Foot Shock), FSW (Forced Swimm), RS (Restraint Stress), TS (Tail Susspension)  
**Timepoints:** Acute, 7D, 14D, 21D  
**Animals:** F1–F5 (all female)

In [ ]:
# Install dependencies if needed
%pip install matplotlib_venn statannotations boruta scikit-learn imbalanced-learn shap tqdm seaborn
%pip install statsmodels scikit-learn matplotlib seaborn tqdm
%pip install torch torch-geometric   
%pip install brainglobe-heatmap brainglobe-atlasapi
%pip install networkx


## Setup

In [ ]:
import os, sys, importlib
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import glob

# Import adapted modules (place all .py files in the same directory or add to sys.path)
import loaderFunctions   as lf
import initFunctions     as init
import analysisFunctions as af
import classifyFunctions as cf
import plotFunctions     as pf
import helperFunctions   as hf
import configFunctions   as config
import pickle
from brainrender_plotRegions import plot_brain_regions

print('All modules loaded successfully.')

## Configuration

In [ ]:
# ── Path to your data CSV ────────────────────────────────────────────────────
CSV_PATH = 'C:\\Users\\LocalAdmin\\Linus\\claude_try\\Data\\merged_wide_with_acro.csv'   # change if located elsewhere

# ── Create output directories ────────────────────────────────────────────────
dirDict = init.setPath_createDirs()

# ── Figure settings ──────────────────────────────────────────────────────────
config.setup_figure_settings()

# ── Heatmap settings ─────────────────────────────────────────────────────────
heatmapDict = config.return_heatmapDict()

# ── Classifier settings (testing = fast, default = full run) ─────────────────
#classifyDict = config.return_classifyDict_testing()   # quick test
classifyDict = config.return_classifyDict_default()   # full run

# ── Plot switches ─────────────────────────────────────────────────────────────
plotDict = config.return_plotDict()

print('Configuration complete.')
print('Output directory:', dirDict['outDir'])

## Load Data

In [ ]:
# Load the wide-format CSV → long-format DataFrame
lightsheet_data = lf.load_stressor_data(CSV_PATH)

print('\nData shape:', lightsheet_data.shape)
print('Columns:', list(lightsheet_data.columns))
lightsheet_data.head(3)

In [ ]:
# Quick sanity check – count samples per stressor and timepoint
print('Samples per stressor:')
print(lightsheet_data.groupby('stressor')['dataset'].nunique())

print('\nSamples per timepoint:')
print(lightsheet_data.groupby('timepoint')['dataset'].nunique())

print('\nSamples per stressor × timepoint:')
print(lightsheet_data.groupby(['stressor','timepoint'])['dataset'].nunique().unstack(fill_value=0))

## Stressor vs Control: Per-region Statistics
(Equivalent to Figure 2 in the original paper)

In [ ]:
importlib.reload(af)

# Full dataset: stressor list (all timepoints pooled)
stressorList = ['Ctrl', 'FS', 'FSW', 'RS', 'TS']

# Returns:
#   stressor_pair_db   – mean percent-change per region for each Ctrl-vs-X comparison
#   stressor_stats_all – descriptive stats per region
#   stressor_pair_data – raw pairwise comparison data (for CIs)
#   stressor_comp_names – e.g. ['Ctrl-FS', 'Ctrl-FSW', 'Ctrl-RS', 'Ctrl-TS']

stressor_pair_db, stressor_stats_all, stressor_pair_data, stressor_comp_names = \
    af.stressor_stats_and_changes(lightsheet_data, stressorList, ctrlSwitch=True)

print('Comparisons computed:', stressor_comp_names)
stressor_pair_db.head()

In [ ]:
# ── Per-timepoint analysis ───────────────────────────────────────────────────
# Some timepoints don't have all stressors, so filter accordingly.

for tp in ['Acute', '7D', '14D', '21D']:
    data_tp = lf.filter_by_timepoint(lightsheet_data, tp)
    stressors_in_tp = data_tp['stressor'].dropna().unique().tolist()
    print(f"Timepoint {tp}: {stressors_in_tp}")

In [ ]:
# ── Whole-brain cFos density heatmap (stressor × timepoint) ──────────────────
from stressor_heatmap import plot_density_heatmap

fig_density = plot_density_heatmap(
    csv_path = CSV_PATH,                   # your existing csv_path variable
    out_dir  = dirDict ["outDir"],          # saves alongside other figures
    scale    = 1e10,                       # multiplies raw values for display
    vmax_pct = 99,                         # saturate top 1% of values
)
plt.show()
plt.close()

## Figure 3: Low-Dimensional Embedding (PCA / LDA)
(Visualise how stressor groups separate in feature space)

In [ ]:
importlib.reload(pf)
importlib.reload(config)
config.setup_LDA_settings()

# Full dataset LDA/PCA across all stressors
ldaDict = {}   # empty dict → use all data for both train and test

pf.plotLowDimEmbed(
    pandasdf    = lightsheet_data,
    column2Plot = 'density_norm',
    dirDict     = dirDict,
    dimRedMeth  = 'PCA',       # 'PCA' or 'LDA'
    classifyDict= classifyDict,
    ldaDict     = ldaDict
)

In [ ]:
importlib.reload(pf)
dist_df, evr = pf.compute_pca_centroid_distances(lightsheet_data, 'density_norm', classifyDict)

## Figure 4a: Classification – All Stressors vs Control
(Confusion matrix + PR curves for each pairwise comparison)

In [ ]:
importlib.reload(cf)
importlib.reload(config)
config.setup_figure_settings()

# ── Pairwise comparisons: Ctrl vs each stressor ───────────────────────────────
# Each label restricts the data to two stressor groups and trains a binary classifier.

plotDict['plot_ConfusionMatrix'] = True
plotDict['plot_PRcurve']         = True
plotDict['plot_SHAPsummary']     = True
plotDict['plot_SHAPcorr']    = True   # False to skip corrmat only
plotDict['shapCorrTop']      = 20     # adjust n regions if needed
plotDict['plot_SHAPforce']       = True
classifyDict['saveLoadswitch']   = True
classifyDict['shuffle']          = True

classifyVec = [
    'class_CtrlFS',     # Control vs Foot Shock
    'class_CtrlFSW',    # Control vs Forced Swim
    'class_CtrlRS',     # Control vs Restraint Stress
    'class_CtrlTS',     # Control vs Tail Suspension
    'class_CtrlAll',    #control vs All pooled
]

for label in classifyVec:
    try:
        print(f"\nClassifying: {label}")
        classifyDict['label'] = label
        cf.classifySamples(lightsheet_data, classifyDict, plotDict, dirDict)
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f"Failed to classify {label}: {e}")

# ── Loop 2: between-stressor with brainglobe ─────────────────────────────────

In [ ]:
# ── Multiclass: all stressors simultaneously ──────────────────────────────────
print("\nClassifying: all stressors (multiclass)")
classifyDict['label'] = 'stressor'
cf.classifySamples(lightsheet_data, classifyDict, plotDict, dirDict)

In [ ]:
# ── Between-stressor comparisons (no Control) ────────────────────────────────
between_stressor_vec = [
    'class_FSRS',     # Foot Shock vs Restraint
    'class_FSTS',     # Foot Shock vs Tail Suspension
    'class_RSTS',     # Restraint vs Tail Suspension
    'class_FSFSW',    # Foot Shock vs Forced Swim
    'class_FSWRS',    # Restraint vs Forced Swim
    'class_FSWTS'     # Tail Suspension vs Forced Swim

]

for label in between_stressor_vec:
    try:
        print(f"\nClassifying: {label}")
        classifyDict['label'] = label
        cf.classifySamples(lightsheet_data, classifyDict, plotDict, dirDict)
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f"Failed to classify {label}: {e}")

     

## Per-Timepoint Classification
(Run the same classifier on each timepoint subset)

In [ ]:
importlib.reload(cf)
plotDict['plot_ConfusionMatrix'] = True
plotDict['plot_PRcurve']         = True
plotDict['plot_SHAPsummary']     = True   # ← add if you want SHAP
plotDict['plot_SHAPforce']       = False  # ← True only for binary classifiers

classifyDict_tp = config.return_classifyDict_default()
classifyDict_tp['shuffle'] = True
TIMEPOINT_ORDER = ['Acute', '7D', '14D', '21D']

for tp in ['Acute', '7D', '14D', '21D']:
    data_tp = lf.filter_by_timepoint(lightsheet_data, tp)
    stressors_in_tp = sorted(data_tp['stressor'].dropna().unique().tolist())

    if len(stressors_in_tp) < 2 or 'Ctrl' not in stressors_in_tp:
        print(f"Skipping {tp}: insufficient stressor groups ({stressors_in_tp})")
        continue

    print(f"\n── Timepoint: {tp} | Stressors: {stressors_in_tp} ──")
    classifyDict_tp['label'] = 'stressor'

    # Create a separate dirDict sub-folder per timepoint
    tp_dirDict = dirDict.copy()
    tp_classifyDir = os.path.join(dirDict['outDir'], f'classif_{tp}')
    tp_tempDir     = os.path.join(dirDict['tempDir'], f'classif_{tp}')
    for d in [tp_classifyDir, tp_tempDir]:
        os.makedirs(d, exist_ok=True)
    tp_dirDict['classifyDir'] = tp_classifyDir
    tp_dirDict['tempDir']     = tp_tempDir

    try:
        cf.classifySamples(data_tp, classifyDict_tp, plotDict, tp_dirDict)
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f"  Failed for {tp}: {e}")

In [ ]:
## Stressor classsification for time effects

In [ ]:
importlib.reload(cf)
plotDict['plot_ConfusionMatrix'] = True
plotDict['plot_PRcurve']         = True
plotDict['plot_SHAPsummary']     = True
plotDict['plot_SHAPforce']       = False  # multiclass → False

classifyDict_str = config.return_classifyDict_default()
classifyDict_str['shuffle']   = True
classifyDict_str['label']     = 'class_tp_all'

TIMEPOINT_ORDER = ['Acute', '7D', '14D', '21D']

for stressor in ['FS', 'FSW', 'RS', 'TS', 'Ctrl']:

    # ── 1. Filter to this stressor ────────────────────────────────────────────
    data_s = lightsheet_data[lightsheet_data['stressor'] == stressor].copy()
    tps_available = [tp for tp in TIMEPOINT_ORDER
                     if tp in data_s['timepoint'].dropna().unique()]

    if len(tps_available) < 2:
        print(f"Skipping {stressor}: only {len(tps_available)} timepoint(s) ({tps_available})")
        continue

    print(f"\n── Stressor: {stressor} | Timepoints: {tps_available} ──")

    # ── 2. Remap so split('_')[0] on dataset index yields the timepoint ───────
    # "FS_7D_F1"  →  "7D_FS_F1"
    data_remap = data_s.copy()
    data_remap['dataset'] = data_s['dataset'].str.replace(
        r'^([^_]+)_([^_]+)_(.+)$', r'\2_\1_\3', regex=True
    )
    # Replace 'stressor' and its backward-compat alias with the timepoint value
    # so that conv_dict {'Acute':'Acute', '7D':'7D', ...} maps correctly.
    data_remap['stressor'] = data_s['timepoint']
    data_remap['drug']     = data_s['timepoint']

    # ── 3. Per-stressor output sub-folder ─────────────────────────────────────
    str_dirDict  = dirDict.copy()
    str_classDir = os.path.join(dirDict['outDir'],  f'classif_tp_{stressor}')
    str_tempDir  = os.path.join(dirDict['tempDir'], f'classif_tp_{stressor}')
    for d in [str_classDir, str_tempDir]:
        os.makedirs(d, exist_ok=True)
    str_dirDict['classifyDir'] = str_classDir
    str_dirDict['tempDir']     = str_tempDir

    try:
        cf.classifySamples(data_remap, classifyDict_str, plotDict, str_dirDict)
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f"  Failed for {stressor}: {e}")

In [ ]:
# ── Pooled timepoint classifier (all stressors combined) ─────────────────────
# FS_21D is missing → 21D has N=20 (4 stressors × 5), others N=25 (5 × 5).
# 'not minority' undersamples Acute/7D/14D to 20 to match 21D.

importlib.reload(cf)

plotDict['plot_ConfusionMatrix'] = True
plotDict['plot_PRcurve']         = True
plotDict['plot_SHAPsummary']     = False
plotDict['plot_SHAPforce']       = False

# Remap: "FS_7D_F1" → "7D_FS_F1", stressor column → timepoint value
data_pool = lightsheet_data[lightsheet_data['stressor'] != 'Ctrl'].copy()
data_pool['dataset']  = data_pool['dataset'].str.replace(
    r'^([^_]+)_([^_]+)_(.+)$', r'\2_\1_\3', regex=True
)
data_pool['stressor'] = data_pool['timepoint']
data_pool['drug']     = data_pool['timepoint']

# Verify class sizes — FS_21D missing should show 21D=20, others=25
print("Samples per timepoint (pooled):")
print(data_pool.groupby('stressor')['dataset'].nunique())

# Output dirs
pool_classDir = os.path.join(dirDict['outDir'],  'classif_tp_pooled')
pool_tempDir  = os.path.join(dirDict['tempDir'], 'classif_tp_pooled')
for d in [pool_classDir, pool_tempDir]:
    os.makedirs(d, exist_ok=True)
pool_dirDict = {**dirDict, 'classifyDir': pool_classDir, 'tempDir': pool_tempDir}

classifyDict_pool = config.return_classifyDict_default()
classifyDict_pool['label']               = 'class_tp_all'
classifyDict_pool['shuffle']             = True
classifyDict_pool['saveLoadswitch']      = True

try:
    cf.classifySamples(data_pool, classifyDict_pool, plotDict, pool_dirDict)
except Exception as e:
    import traceback; traceback.print_exc()
    print(f"  Failed for pooled timepoint: {e}")

## Figure 4b–d: Cross-Comparison Summary
(AUC bar plot, feature violin plot, feature heatmap)

In [ ]:
importlib.reload(pf)
importlib.reload(hf)
importlib.reload(config)
config.setup_figure_settings()

# Define the desired display order of comparisons
sortedNames = [
    'Ctrl vs FS',
    'Ctrl vs FSW',
    'Ctrl vs RS',
    'Ctrl vs TS',
    'FS vs RS',
    'FS vs TS',
    'FS vs FSW',
    'RS vs TS',
    'FSW vs TS',
    'FSW vs RS',
]

filterByFreq = 0.75 * classifyDict['CV_count']

try:
    featureLists, comparisonNames, aucScores, meanScores, accStd, \
    oobErrors, aucPerSplit, aucScrambleScores, meanScrambleScores = \
    hf.retrieve_dict_data(dirDict, sortedNames, classifyDict)
    print('Comparison names found:', comparisonNames)
except AssertionError as e:
    print(f'No saved classifier results found yet: {e}')
    print('Run the classification cells above first.')

In [ ]:
# Figure 4b – Combined AUC / Accuracy / OOB summary plot
try:
    pf.plot_cross_model_AUC(
    comparisonNames, aucScores, aucScrambleScores, dirDict,
    meanScores=meanScores,
    accStd=accStd,
    oobErrors=oobErrors,
    aucPerSplit=aucPerSplit,
    meanScrambleScores=meanScrambleScores
)
except Exception as e:
    import traceback; traceback.print_exc()
    print(f'plot_cross_model_AUC failed: {e}')

In [ ]:
# Figure 4c – Feature count violin plots
try:
    pf.plot_featureCount_violin(comparisonNames, featureLists, dirDict)
except Exception as e:
    print(f'plot_featureCount_violin failed: {e}')

In [ ]:
# Figure 4d – Feature heatmap
try:
    pf.plot_featureHeatMap(lightsheet_data, comparisonNames, featureLists, filterByFreq, dirDict)
except Exception as e:
    print(f'plot_featureHeatMap failed: {e}')

In [ ]:
# ── Timepoint PR-AUC summary (loads from cache — no rerun needed) ────────────
importlib.reload(pf)
config.setup_figure_settings()
 
pf.plot_timepoint_PRAUC_summary(
    dirDict            = dirDict,
    stressor_list      = ['FSW', 'RS', 'TS', 'Ctrl'],
    timepoint_order    = ['Acute', '7D', '14D', '21D'],
    show_mean_marker   = True,   # ★ at mean AUC after last timepoint
    show_per_split_jitter = False, # light scatter of per-split AUCs
    figsize            = (3.5, 3.0),
    save_name          = 'timepoint_PRAUC_summary',
)

In [ ]:
# ── Per-timepoint multiclass PR-AUC summary ──────────────────────────────────
# Reads from classif_Acute / classif_7D / classif_14D / classif_21D caches.
# Each point = PR-AUC for that stressor class in the multiclass classifier
# run on all animals sacrificed at that timepoint.

importlib.reload(pf)
config.setup_figure_settings()

pf.plot_timepoint_multiclass_PRAUC(
    dirDict              = dirDict,
    stressor_list        = ['Ctrl', 'FS', 'FSW', 'RS', 'TS'],
    timepoint_order      = ['Acute', '7D', '14D', '21D'],
    show_mean_line       = True,    # dashed black line = macro mean AUC
    show_per_split_jitter= False,    # light grey dots = per-split mean AUC
    figsize              = (3.5, 3.0),
    save_name            = 'timepoint_multiclass_PRAUC',
)


In [ ]:
importlib.reload(pf)
config.setup_figure_settings()

# ── Option A: load from the full CV cache pkl ─────────────────────────────────
# The pkl lives inside the model subfolder of classif_tp_pooled/tempDir.
# Typical path structure (adjust to your actual path):
#   <tempDir>/classif_tp_pooled/<data_param_string>/<model_param_string>/Real_outdata.pkl

import glob
pool_temp = os.path.join(dirDict['tempDir'], 'classif_tp_pooled')
pkl_candidates = glob.glob(os.path.join(pool_temp, '**', 'Real_outdata.pkl'), recursive=True)
print("Found pkl files:", pkl_candidates)

# Pick the first (or inspect the list and index manually)
pkl_path_pooled = pkl_candidates[0] if pkl_candidates else None

# ── Option B: load from the text readout ─────────────────────────────────────
# The txt lives inside the model's outDir subfolder.
#   <classifyDir>/classif_tp_pooled/<data_param_string>/<model_param_string>/featureSelReadout.txt

pool_out = os.path.join(dirDict['outDir'], 'classif_tp_pooled')
txt_candidates = glob.glob(os.path.join(pool_out, '**', 'featureSelReadout.txt'), recursive=True)
print("Found txt files:", txt_candidates)
txt_path_pooled = txt_candidates[0] if txt_candidates else None

# ── Run the plot ──────────────────────────────────────────────────────────────
# Use pkl if available, fall back to txt
boruta_df = pf.plot_boruta_regions_timepoint(
    csv_path        = CSV_PATH,
    pkl_path        = pkl_path_pooled,   # set to None to use txt instead
    txt_path        = None,              # set txt_path_pooled if no pkl
    dirDict         = dirDict,
    cv_count        = classifyDict['CV_count'],
    min_count       = 75,        # show all selected regions; raise to filter
    group_by_area   = True,     # cluster bars by brain area
    show_fraction   = True,     # x-axis = fraction of folds (0-1)
    save_name       = 'boruta_regions_timepoint_pooled',
)

# Inspect the DataFrame
if boruta_df is not None:
    print(boruta_df.sort_values('count', ascending=False).to_string(index=False))

In [ ]:
from feature_heatmap import plot_classifier_heatmaps

# One SVG/PNG per brain hyperstructure
plot_classifier_heatmaps(
    lightsheet_data  = lightsheet_data,
    comparisonNames  = comparisonNames,
    featureLists     = featureLists,
    filterByFreq     = filterByFreq,          # e.g. 0.75 * classifyDict['CV_count']
    dirDict          = dirDict,
    cv_count         = classifyDict['CV_count'],
    normalize        = True,                  # values shown as % of CV folds
    hyperstructures  = None,                 # None → all 10 areas; or e.g. ['Isocortex','Thalamus']
)

# Or a single combined multi-panel figure (good for supplementary)
from feature_heatmap import plot_classifier_heatmaps_combined
plot_classifier_heatmaps_combined(
    lightsheet_data  = lightsheet_data,
    comparisonNames  = comparisonNames,
    featureLists     = featureLists,
    filterByFreq     = filterByFreq,          # e.g. 0.75 * classifyDict['CV_count']
    dirDict          = dirDict,
    cv_count         = classifyDict['CV_count'],
    hyperstructures= None,                 # None → all 10 areas; or e.g. ['Isocortex','Thalamus']
    normalize        = True,
    n_cols = 2,
)

In [ ]:
## brainrencer call


In [ ]:
import importlib
import brainrender_plotRegions as br
importlib.reload(br)
from brainrender_plotRegions import generate_brainrender_csvs, plot_brain_regions
from brainrender_plotRegions import plot_brain_regions

br_dir = os.path.join(dirDict['outDir'], 'brainrender')
classifyDict_full = config.return_classifyDict_default()
generate_brainrender_csvs(lightsheet_data, dirDict, classifyDict_full, output_dir=br_dir)
plot_brain_regions(csv_dir=br_dir, output_dir=br_dir)

In [ ]:
import importlib
import brainrender_plotRegions as br
importlib.reload(br)
from brainrender_plotRegions import generate_brainrender_csvs, plot_brain_regions
from brainrender_plotRegions import plot_brain_regions
br_dir = os.path.join(dirDict['outDir'], 'brainrender')
plot_brain_regions(csv_dir=br_dir, output_dir=br_dir)


In [ ]:
## Statistical real vs shuffeld data
# calls function in helperFunctions.py

In [ ]:
import importlib
import helperFunctions as hf
importlib.reload(hf)

stats_df = hf.compute_permutation_stats(dirDict, tagList=None, alpha=0.05)

# Display
with pd.option_context('display.float_format', '{:.4f}'.format, 'display.max_rows', 60):
    display(stats_df[[
        'classifier', 'mean_auc_real', 'mean_auc_shuffle', 'auc_delta',
        'p_value', 'p_adj', 'significant'
    ]])

# Optional: save to CSV
stats_df.to_csv(os.path.join(dirDict['outDir'], 'permutation_stats.csv'), index=False)

## SHAP Analysis – Most Discriminative Comparison
(Equivalent to Figure 5 in the original paper)

In [ ]:
importlib.reload(cf)

plotDict['plot_ConfusionMatrix'] = True
plotDict['plot_PRcurve']         = True
plotDict['plot_SHAPsummary']     = True
plotDict['plot_SHAPforce']       = True

# Run SHAP for a specific comparison (change label as desired)
classifyDict['label'] = 'class_CtrlFS'   # e.g. Ctrl vs Foot Shock

cf.classifySamples(lightsheet_data, classifyDict, plotDict, dirDict)

## Supplemental: Per-Region Delta Plot
(Equivalent to Supplemental Figure 2 in the original paper)

In [ ]:
importlib.reload(pf)
importlib.reload(config)

config.setup_figure_changeFonts(6)
config.setup_saldiff_settings()

stressorList_plot = ['Ctrl', 'FS', 'FSW', 'RS', 'TS']

stressor_pair_db, stressor_stats_all, stressor_pair_data, stressor_comp_names = \
    af.stressor_stats_and_changes(lightsheet_data, stressorList_plot, ctrlSwitch=True)

fileOutName = os.path.join(dirDict['outDir'], 'fig_deltaPerRegion_Ctrl')
if len(glob.glob(fileOutName + '.*')) == 0:
    try:
        pf.plot_cFos_delta_new(
            lightsheet_data,
            stressor_pair_data,
            stressor_comp_names,
            stressorList_plot,
            fileOutName
        )
    except Exception as e:
        print(f'plot_cFos_delta_new failed: {e}')
else:
    print('Delta plot already exists:', fileOutName)

In [ ]:
## Anova using statsmodel and GNN using torch-geometric 

In [ ]:
from anova_pipeline import run_pipeline, CFG
run_pipeline(CFG)

In [ ]:
##Time × Stressor modelling for cFos lightsheet data.

In [ ]:
import modelFunctions as mf
results = mf.run_full_pipeline('merged_wide_with_acro.csv', out_dir='./model_out')
# Or step by step:
#long_df = mf.melt_to_long(df)
#results = mf.run_models(long_df)
#table   = mf.get_ranked_table(results, effect='interact', top_n=30)